# ERGT Colab Phase 3 Runner

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

Default mode is `pilot`: it runs the four ERGT-v1 pilot conditions and compares them against the tracked pilot baseline report. Switch `RUN_MODE` to `proof` only after the pilot path is healthy.

In [ ]:
# Colab run controls
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

# Use "pilot" first. Use "proof" after pilot artifacts look healthy.
RUN_MODE = "pilot"  # "pilot" or "proof"

# Baseline pilot is already tracked in the repository. Set True to re-run it on Colab.
RUN_BASELINE = False
RUN_ERGT = True
RUN_COMPARISON = True

# Gate is intended for proof runs. Keep False for pilot health checks.
RUN_GATE = False

DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
print("COLAB_TPU_ADDR:", os.environ.get("COLAB_TPU_ADDR"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, set Hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

x = torch.randn(1024, 1024, device="cuda")
y = x @ x
print("cuda_tensor_test:", float(y.mean().detach().cpu()))

## 2. Clone or update repository and install dependencies

In [ ]:
from pathlib import Path

project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print("repo:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"], check=True)

## 3. Prepare WikiText-2 data

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    check=True,
)

## 4. Define Phase 3 commands

In [ ]:
if RUN_MODE not in {"pilot", "proof"}:
    raise ValueError("RUN_MODE must be 'pilot' or 'proof'")

if RUN_MODE == "pilot":
    baseline_config = "configs/baseline/pilot_wikitext2.json"
    baseline_results = "runs/phase0_baseline/pilot_wikitext2/baseline_results.json"
    ergt_configs = {
        "alpha_zero": "configs/ergt_v1/pilot_alpha_zero.json",
        "real_d": "configs/ergt_v1/pilot_real_d.json",
        "random_d": "configs/ergt_v1/pilot_random_d.json",
        "shuffled_d": "configs/ergt_v1/pilot_shuffled_d.json",
    }
    ergt_results = {
        "alpha_zero": "runs/phase3_geo_attention/pilot_alpha_zero/metrics.json",
        "real_d": "runs/phase3_geo_attention/pilot_real_d/metrics.json",
        "random_d": "runs/phase3_geo_attention/pilot_random_d/metrics.json",
        "shuffled_d": "runs/phase3_geo_attention/pilot_shuffled_d/metrics.json",
    }
    comparison_dir = "runs/phase3_geo_attention/pilot_comparison"
else:
    baseline_config = "configs/baseline/proof_wikitext2.json"
    baseline_results = "runs/phase0_baseline/proof_wikitext2/baseline_results.json"
    ergt_configs = {
        "alpha_zero": "configs/ergt_v1/alpha_zero.json",
        "real_d": "configs/ergt_v1/real_d.json",
        "random_d": "configs/ergt_v1/random_d.json",
        "shuffled_d": "configs/ergt_v1/shuffled_d.json",
    }
    ergt_results = {
        "alpha_zero": "runs/phase3_geo_attention/alpha_zero/metrics.json",
        "real_d": "runs/phase3_geo_attention/real_d/metrics.json",
        "random_d": "runs/phase3_geo_attention/random_d/metrics.json",
        "shuffled_d": "runs/phase3_geo_attention/shuffled_d/metrics.json",
    }
    comparison_dir = "runs/phase3_geo_attention"

def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

print("RUN_MODE:", RUN_MODE)
print("baseline_results:", baseline_results)
print("comparison_dir:", comparison_dir)

## 5. Run baseline if requested

In [ ]:
if RUN_BASELINE:
    run_command(
        [
            sys.executable,
            "experiments/train_baseline.py",
            "--config",
            baseline_config,
            "--data-dir",
            DATA_DIR,
            "--device",
            DEVICE,
        ]
    )
else:
    print("Skipping baseline run. Existing file required:", baseline_results)
    if RUN_COMPARISON and not Path(baseline_results).exists():
        raise FileNotFoundError(
            f"Missing {baseline_results}. Set RUN_BASELINE=True or upload the report."
        )

## 6. Run ERGT-v1 Phase 3 conditions

In [ ]:
if RUN_ERGT:
    for condition, config_path in ergt_configs.items():
        print("\n=== ERGT condition:", condition, "===")
        run_command(
            [
                sys.executable,
                "experiments/train_ergt_v1.py",
                "--config",
                config_path,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("Skipping ERGT condition runs.")

## 7. Compare Phase 3 conditions

In [ ]:
if RUN_COMPARISON:
    for path in [baseline_results, *ergt_results.values()]:
        if not Path(path).exists():
            raise FileNotFoundError(f"Missing required metrics file: {path}")

    run_command(
        [
            sys.executable,
            "experiments/compare_phase3.py",
            "--baseline",
            baseline_results,
            "--alpha-zero",
            ergt_results["alpha_zero"],
            "--real-d",
            ergt_results["real_d"],
            "--random-d",
            ergt_results["random_d"],
            "--shuffled-d",
            ergt_results["shuffled_d"],
            "--output-dir",
            comparison_dir,
        ]
    )
else:
    print("Skipping comparison.")

## 8. Optional Gate 1 decision

In [ ]:
if RUN_GATE:
    if RUN_MODE != "proof":
        raise RuntimeError("Gate 1 should be run on proof outputs, not pilot outputs.")
    run_command(
        [
            sys.executable,
            "-m",
            "evaluation.gate_decision",
            "--comparison",
            "runs/phase3_geo_attention/comparison_results.json",
            "--ablation",
            "runs/phase3_geo_attention/ablation_report.json",
            "--output",
            "runs/gates/phase3_to_phase4_decision.json",
        ]
    )
else:
    print("Skipping Gate 1 decision.")

## 9. Print key JSON outputs

In [ ]:
def print_json(path):
    path = Path(path)
    print("\n===", path, "===")
    if not path.exists():
        print("missing")
        return
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    print(json.dumps(payload, indent=2, sort_keys=True)[:8000])

print_json(baseline_results)
for path in ergt_results.values():
    print_json(path)
print_json(str(Path(comparison_dir) / "comparison_results.json"))
print_json(str(Path(comparison_dir) / "ablation_report.json"))
if RUN_GATE:
    print_json("runs/gates/phase3_to_phase4_decision.json")

## 10. Archive outputs for upload

In [ ]:
archive_base = f"/content/ergt_{RUN_MODE}_runs"
archive_path = shutil.make_archive(archive_base, "zip", PROJECT_DIR, "runs")
print("Archive ready:", archive_path)
print("Download this zip and share the JSON/log outputs for review.")